In [102]:
# =============================================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# =============================================================================
import os
import sys
import warnings
from pathlib import Path

# Mengabaikan peringatan internal numerik & pandas
warnings.filterwarnings('ignore')

# Pustaka manipulasi data & array
import pandas as pd
import numpy as np

# Monkey-patch untuk kompatibilitas Pandas 2.x/3.x dengan pustaka eksternal
pd.DataFrame.iteritems = pd.DataFrame.items

# Pustaka teknikal & framework backtesting standar industri
import pandas_ta as ta
from backtesting import Backtest, Strategy
from backtesting.lib import FractionalBacktest

# Pustaka ekspor & penataan gaya Excel
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Konfigurasi format tampilan angka desimal pada Pandas (2 desimal)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Environment Setup: Seluruh pustaka (pandas, numpy, pandas_ta, backtesting, openpyxl) berhasil dimuat.")


Environment Setup: Seluruh pustaka (pandas, numpy, pandas_ta, backtesting, openpyxl) berhasil dimuat.


In [103]:
# =============================================================================
# CELL 2: GLOBAL CONFIGURATION & TARGET PATHS
# =============================================================================
# Konfigurasi parameter kuantitatif simetris penuh (Identik antara Long dan Short)
GLOBAL_PARAMS = {
    'initial_capital': 1000.0,
    'fee': 0.0003,                    # Komisi bursa 0.03% per sisi transaksi
    'ema_macro_fast': 21,              # 1 bulan kalender bursa (Zakamulin, 2017)
    'ema_macro_slow': 63,              # 1 kuartal kalender bursa (Zakamulin, 2017)
    'macro_spread_threshold': 0.020,   # Ambang batas band 1.0% (Brock et al., 1992), def: 0.010
    'ema_micro': 33,                   # 1 minggu bursa intraday (5 hari x 6.5 jam)
    'atr_period': 7,                  # Wilder's ATR standar (Wilder, 1978), def: 14
    'atr_mult': 6.0,                   # Toleransi volatilitas lebar (Kaminski & Lo, 2014)
    'cooldown_bars': 12,                # Jeda 1 hari perdagangan penuh (6.5 jam)
    'min_profit_pct': 0.010            # Aktivasi ratchet trailing setelah profit 1.0%
}

# Daftar instrumen acuan yang diuji secara batch
ASSETS = ['SPY', 'QQQ', 'DJI', 'AAPL']

# Jalur direktori kerja dan berkas output Excel target (E:/kultivasi_4_tahun/ta/)
BASE_DIR = Path("E:/kultivasi_4_tahun/ta")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd()

# Jalur pencarian data mentah adaptif
DATA_DIR = BASE_DIR / "data_index"
if not DATA_DIR.exists():
    DATA_DIR = BASE_DIR.parent / "TugasAkhir" / "data_index"

# Berkas target luaran Excel di direktori target ta/
FILE_RINGKASAN = BASE_DIR / "Ringkasan_Kinerja_Semua_Instrumen.xlsx"
FILE_TRADES = BASE_DIR / "Detail_Trades_Semua_Instrumen.xlsx"

print("Konfigurasi Parameter Global (Simetris):")
for k, v in GLOBAL_PARAMS.items():
    print(f"  * {k:<25}: {v}")
print(f"\nDirektori Target : {BASE_DIR}")
print(f"Berkas Ringkasan : {FILE_RINGKASAN}")
print(f"Berkas Trades    : {FILE_TRADES}")


Konfigurasi Parameter Global (Simetris):
  * initial_capital          : 1000.0
  * fee                      : 0.0003
  * ema_macro_fast           : 21
  * ema_macro_slow           : 63
  * macro_spread_threshold   : 0.02
  * ema_micro                : 33
  * atr_period               : 7
  * atr_mult                 : 6.0
  * cooldown_bars            : 12
  * min_profit_pct           : 0.01

Direktori Target : d:\Kuliah\TA\ta
Berkas Ringkasan : d:\Kuliah\TA\ta\Ringkasan_Kinerja_Semua_Instrumen.xlsx
Berkas Trades    : d:\Kuliah\TA\ta\Detail_Trades_Semua_Instrumen.xlsx


In [104]:
# =============================================================================
# CELL 3: DATA PREPROCESSING ENGINE
# =============================================================================
def calc_ema_series(series, span):
    """Menghitung Exponential Moving Average (EMA) berbasis span."""
    return pd.Series(series).ewm(span=span, adjust=False).mean().values

def calc_atr_series(high, low, close, period=14):
    """Menghitung Average True Range (ATR) berbasis True Range Wilder/EMA."""
    high_s = pd.Series(high)
    low_s = pd.Series(low)
    close_s = pd.Series(close)
    close_prev = close_s.shift(1)
    tr = pd.concat([
        high_s - low_s,
        (high_s - close_prev).abs(),
        (low_s - close_prev).abs()
    ], axis=1).max(axis=1)
    return pd.Series(tr).ewm(span=period, adjust=False).mean().values

def load_clean_data(file_path, is_1h=False):
    """Membaca data CSV/Parquet, sanitasi jam RTH 09:00-16:00, dan eliminasi akhir pekan."""
    p = Path(file_path)
    if not p.exists():
        raise FileNotFoundError(f"Berkas data tidak ditemukan pada path: {p}")
        
    if p.suffix.lower() == '.parquet':
        df = pd.read_parquet(p)
    else:
        with open(p, 'r', encoding='utf-8') as f:
            first_line = f.readline()
            sep = '\t' if '\t' in first_line else ','
        df = pd.read_csv(p, sep=sep)
        
    df.columns = [c.lower() for c in df.columns]
    
    # Standarisasi kolom waktu
    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'])
    elif 'date' in df.columns:
        df['datetime'] = pd.to_datetime(df['date'])
        df.drop(columns=['date'], inplace=True, errors='ignore')
    elif 'time' in df.columns:
        df['datetime'] = pd.to_datetime(df['time'])
        
    df.sort_values('datetime', inplace=True)
    df.set_index('datetime', inplace=True)
    
    # Eliminasi akhir pekan (Sabtu=5, Minggu=6)
    df = df[df.index.dayofweek < 5]
    
    # Filter jam bursa reguler (Regular Trading Hours 09:00 - 16:00 EST)
    if is_1h:
        df = df[(df.index.hour >= 9) & (df.index.hour <= 16)]
        
    df['date'] = df.index.date
    return df

def get_asset_file_paths(asset):
    """Pemetaan adaptif berkas data 1D dan 1H per instrumen."""
    search_dirs = [DATA_DIR, BASE_DIR / "data_index", BASE_DIR.parent / "TugasAkhir" / "data_index"]
    for d in search_dirs:
        mapping = {
            'SPY': {'1d': d / 'spy_US500_D1.parquet', '1h': d / 'spy_US500_H1.parquet'},
            'QQQ': {'1d': d / 'Nasdaq100_1d_data.csv', '1h': d / 'Nasdaq100_1h_data.csv'},
            'DJI': {'1d': d / 'Dow_Jones_1d_data.csv', '1h': d / 'Dow_Jones_1h_data.csv'},
            'AAPL': {'1d': d / 'validasi-aapl' / 'aapl_1d.csv', '1h': d / 'validasi-aapl' / 'aapl_1h.csv'}
        }
        if mapping[asset]['1d'].exists() and mapping[asset]['1h'].exists():
            return mapping[asset]
            
    raise FileNotFoundError(f"Tidak dapat menemukan berkas data lengkap untuk instrumen: {asset}")

def preprocess_asset_data(asset, start_dt='2018-02-19', end_dt='2024-05-10 23:59:59'):
    """
    Pipeline Feature Engineering Multi-Timeframe:
    1. Menghitung indikator 1D secara utuh pada data penuh (Full Period) dengan pergeseran 1 bar (.shift(1))
       guna mencegah kebocoran data masa depan (Zero Lookahead Bias).
    2. Menyaring data 1H pada rentang waktu evaluasi seragam (2018-2024).
    3. Menggabungkan indikator 1D ke 1H berbasis tanggal.
    4. Menghitung EMA Mikro 32 dan ATR 14 pada 1H.
    """
    paths = get_asset_file_paths(asset)
    d1 = load_clean_data(paths['1d'], is_1h=False)
    h1 = load_clean_data(paths['1h'], is_1h=True)
    
    # 1. Feature Engineering Dimensi Makro 1D (Shift 1 bar untuk kausalitas)
    d1['ema_fast'] = pd.Series(calc_ema_series(d1['close'], GLOBAL_PARAMS['ema_macro_fast']), index=d1.index).shift(1)
    d1['ema_slow'] = pd.Series(calc_ema_series(d1['close'], GLOBAL_PARAMS['ema_macro_slow']), index=d1.index).shift(1)
    thresh = GLOBAL_PARAMS['macro_spread_threshold']
    
    # Aturan Makro Simetris:
    d1['macro_bullish'] = d1['ema_fast'] > (d1['ema_slow'] * (1.0 + thresh))
    d1['macro_bearish'] = d1['ema_fast'] < (d1['ema_slow'] * (1.0 - thresh))
    d1['macro_bull_alive'] = d1['ema_fast'] >= d1['ema_slow']
    d1['macro_bear_alive'] = d1['ema_fast'] <= d1['ema_slow']
    
    # 2. Sinkronisasi rentang waktu evaluasi bersama
    h1 = h1[(h1.index >= pd.to_datetime(start_dt)) & (h1.index <= pd.to_datetime(end_dt))]
    
    # 3. Penyelarasan MTF 1D ke 1H berbasis tanggal
    d1_sub = d1[['date', 'macro_bullish', 'macro_bearish', 'macro_bull_alive', 'macro_bear_alive']]
    df = pd.merge(h1.reset_index(), d1_sub, on='date', how='left')
    df.ffill(inplace=True)
    df.set_index('datetime', inplace=True)
    
    # 4. Standardisasi format kolom kapital sesuai spesifikasi Backtesting.py
    col_rename = {'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'}
    df.rename(columns=col_rename, inplace=True)
    
    # 5. Feature Engineering Dimensi Mikro 1H
    df['EMA_micro'] = calc_ema_series(df['Close'], GLOBAL_PARAMS['ema_micro'])
    df['ATR'] = calc_atr_series(df['High'], df['Low'], df['Close'], GLOBAL_PARAMS['atr_period'])
    df.dropna(inplace=True)
    
    return df

print("Data Preprocessing Engine berhasil didefinisikan.")


Data Preprocessing Engine berhasil didefinisikan.


In [105]:
# =============================================================================
# CELL 4: UNIFIED STRATEGY CLASS (SymmetricMTFStrategy)
# =============================================================================
class SymmetricMTFStrategy(Strategy):
    """
    Strategi Kuantitatif Multi-Timeframe Simetris Penuh.
    Mendukung tiga mode operasi melalui atribut 'mode':
      - 'long_only'  : Hanya mengeksekusi posisi Long saat tren makro Bullish.
      - 'short_only' : Hanya mengeksekusi posisi Short saat tren makro Bearish.
      - 'combined'   : Mengeksekusi kedua arah secara riil di dalam engine simulasi.
    """
    mode = 'combined' # 'long_only', 'short_only', atau 'combined'
    atr_mult = GLOBAL_PARAMS['atr_mult']
    cooldown_bars = GLOBAL_PARAMS['cooldown_bars']
    min_profit_pct = GLOBAL_PARAMS['min_profit_pct']

    def init(self):
        # Registrasi indikator teknikal melalui self.I() agar terpantau engine
        self.ema_micro = self.I(calc_ema_series, self.data.Close, GLOBAL_PARAMS['ema_micro'], overlay=True)
        self.atr = self.I(calc_atr_series, self.data.High, self.data.Low, self.data.Close, GLOBAL_PARAMS['atr_period'], overlay=False)
        self.macro_bullish = self.I(lambda: self.data.df['macro_bullish'].values, overlay=False)
        self.macro_bearish = self.I(lambda: self.data.df['macro_bearish'].values, overlay=False)
        self.macro_bull_alive = self.I(lambda: self.data.df['macro_bull_alive'].values, overlay=False)
        self.macro_bear_alive = self.I(lambda: self.data.df['macro_bear_alive'].values, overlay=False)

        # Pelacak internal status posisi dan order
        self.bars_since_exit = 999
        self.trailing_sl = 0.0
        self.entry_price = 0.0
        self.trade_records = []
        self.active_record = None

    def next(self):
        # Warm-up period indikator mikro
        if len(self.data) < GLOBAL_PARAMS['ema_micro']:
            return

        # 1. Logika Pembukaan Posisi Baru (Flat State)
        if not self.position:
            self.bars_since_exit += 1
            if self.bars_since_exit >= self.cooldown_bars:
                long_allowed = (self.mode in ['long_only', 'combined'])
                short_allowed = (self.mode in ['short_only', 'combined'])

                # Logika Sinyal Entry Simetris dengan Konfirmasi Kemiringan EMA Mikro:
                # Long : Makro Bullish (1D) + Pullback Retest (Low bar-2 <= EMA33) + Rebound (Close bar-1 > EMA33) + Slope EMA Mikro Naik
                long_trigger = (
                    long_allowed and
                    self.macro_bullish[-1] and
                    (self.data.Low[-2] <= self.ema_micro[-2]) and
                    (self.data.Close[-1] > self.ema_micro[-1]) and
                    (self.ema_micro[-1] >= self.ema_micro[-2])
                )

                # Short: Makro Bearish (1D) + Retest Atas (High bar-2 >= EMA33) + Reject (Close bar-1 < EMA33) + Slope EMA Mikro Turun
                short_trigger = (
                    short_allowed and
                    self.macro_bearish[-1] and
                    (self.data.High[-2] >= self.ema_micro[-2]) and
                    (self.data.Close[-1] < self.ema_micro[-1]) and
                    (self.ema_micro[-1] <= self.ema_micro[-2])
                )

                if long_trigger:
                    self.buy(size=0.9999)
                    self.entry_price = self.data.Close[-1]
                    self.trailing_sl = self.entry_price - (self.atr_mult * self.atr[-1])
                    self.active_record = {
                        'EntryBar': len(self.data) - 1,
                        'EntryTime': self.data.index[-1],
                        'Mode': self.mode,
                        'Position_Type': 'LONG',
                        'ATR_At_Entry': self.atr[-1],
                        'Trailing_SL_Final': self.trailing_sl,
                        'Exit_Reason': None
                    }
                    return
                elif short_trigger:
                    self.sell(size=0.9999)
                    self.entry_price = self.data.Close[-1]
                    self.trailing_sl = self.entry_price + (self.atr_mult * self.atr[-1])
                    self.active_record = {
                        'EntryBar': len(self.data) - 1,
                        'EntryTime': self.data.index[-1],
                        'Mode': self.mode,
                        'Position_Type': 'SHORT',
                        'ATR_At_Entry': self.atr[-1],
                        'Trailing_SL_Final': self.trailing_sl,
                        'Exit_Reason': None
                    }
                    return

        # 2. Logika Manajemen Posisi Berjalan & Mekanisme Exit
        else:
            if self.position.is_long:
                # Perbarui Ratchet Trailing SL
                new_sl = self.data.Close[-1] - (self.atr_mult * self.atr[-1])
                self.trailing_sl = max(self.trailing_sl, new_sl)
                if self.active_record:
                    self.active_record['Trailing_SL_Final'] = self.trailing_sl

                # Cek ambang profit minimum
                is_in_profit = self.data.Close[-1] >= (self.entry_price * (1.0 + self.min_profit_pct))

                # Exit 1: Trailing Stop Terpicu (Pengunci Keuntungan saat profit tercapai)
                if is_in_profit and (self.data.Close[-1] <= self.trailing_sl):
                    if self.active_record:
                        self.active_record['Exit_Reason'] = 'Trailing Stop Hit'
                        self.trade_records.append(self.active_record.copy())
                        self.active_record = None
                    self.position.close()
                    self.bars_since_exit = 0
                    return

                # Exit 2: Hard-Exit Pembalikan Tren Makro (EMA 21 < EMA 63)
                if not self.macro_bull_alive[-1]:
                    if self.active_record:
                        self.active_record['Exit_Reason'] = 'Macro Trend Reversal'
                        self.trade_records.append(self.active_record.copy())
                        self.active_record = None
                    self.position.close()
                    self.bars_since_exit = 0
                    return

            elif self.position.is_short:
                # Perbarui Ratchet Trailing SL
                new_sl = self.data.Close[-1] + (self.atr_mult * self.atr[-1])
                self.trailing_sl = min(self.trailing_sl, new_sl)
                if self.active_record:
                    self.active_record['Trailing_SL_Final'] = self.trailing_sl

                # Exit 1: Stop Loss / Trailing Stop Terpicu (Disiplin memotong risiko posisi short)
                if self.data.Close[-1] >= self.trailing_sl:
                    if self.active_record:
                        self.active_record['Exit_Reason'] = 'Trailing Stop Hit'
                        self.trade_records.append(self.active_record.copy())
                        self.active_record = None
                    self.position.close()
                    self.bars_since_exit = 0
                    return

                # Exit 2: Hard-Exit Pembalikan Tren Makro (EMA 21 > EMA 63)
                if not self.macro_bear_alive[-1]:
                    if self.active_record:
                        self.active_record['Exit_Reason'] = 'Macro Trend Reversal'
                        self.trade_records.append(self.active_record.copy())
                        self.active_record = None
                    self.position.close()
                    self.bars_since_exit = 0
                    return

print("Unified Strategy Class (SymmetricMTFStrategy) berhasil didefinisikan.")


Unified Strategy Class (SymmetricMTFStrategy) berhasil didefinisikan.


In [106]:
# =============================================================================
# CELL 5: TRADE RECONCILIATION & CLEANING FUNCTION
# =============================================================================
def reconcile_trades(stats, ohlc_df, mode_label, fractional_unit=1e-4):
    """
    Rekonsiliasi log transaksi riil:
    1. Menggabungkan stats['_trades'] dengan pelacak internal Strategy.
    2. Menghitung MAE, MFE, dan Profit Giveback murni dari irisan OHLC intra-trade riil.
    3. Menjamin zero NaN dan menyusun 18 kolom standar institusional.
    """
    trades = stats['_trades'].copy()
    if len(trades) == 0:
        return pd.DataFrame(columns=[
            'Trade_ID', 'Mode', 'Position_Type', 'Size', 'EntryTime', 'ExitTime',
            'Duration', 'Bars_Held', 'EntryPrice', 'ExitPrice', 'Trailing_SL_Final',
            'ATR_At_Entry', 'PnL_USD', 'Return_Pct', 'MAE_Pct', 'MFE_Pct',
            'Profit_Giveback_Pct', 'Exit_Reason'
        ])
    
    strat = stats['_strategy']
    records = list(strat.trade_records)
    
    # Tangani transaksi yang ditutup otomatis pada akhir rentang data oleh engine
    if strat.active_record is not None and len(records) < len(trades):
        rec_copy = strat.active_record.copy()
        rec_copy['Exit_Reason'] = 'End of Data'
        records.append(rec_copy)
        
    trade_rows = []
    
    for i in range(len(trades)):
        tr = trades.iloc[i]
        entry_bar = int(tr['EntryBar'])
        exit_bar = int(tr['ExitBar'])
        pos_size = float(tr['Size'])
        is_long = pos_size > 0
        pos_type = 'LONG' if is_long else 'SHORT'
        
        entry_px = float(tr['EntryPrice'])
        exit_px = float(tr['ExitPrice'])
        pnl = float(tr['PnL'])
        ret_pct = float(tr['ReturnPct']) * 100.0
        
        bars_held = exit_bar - entry_bar
        
        # Irisan matriks harga intra-trade deterministik (real OHLC slice)
        trade_slice = ohlc_df.iloc[entry_bar : exit_bar + 1]
        
        if is_long:
            lowest_price = float(trade_slice['Low'].min())
            mae_usd = min(0.0, lowest_price - entry_px)
            mae_pct = (mae_usd / entry_px) * 100.0
            
            highest_price = float(trade_slice['High'].max())
            mfe_usd = max(0.0, highest_price - entry_px)
            mfe_pct = (mfe_usd / entry_px) * 100.0
            
            if highest_price > entry_px:
                profit_giveback_pct = ((exit_px - highest_price) / highest_price) * 100.0
            else:
                profit_giveback_pct = 0.0
        else:
            highest_price = float(trade_slice['High'].max())
            mae_usd = min(0.0, entry_px - highest_price)
            mae_pct = (mae_usd / entry_px) * 100.0
            
            lowest_price = float(trade_slice['Low'].min())
            mfe_usd = max(0.0, entry_px - lowest_price)
            mfe_pct = (mfe_usd / entry_px) * 100.0
            
            if lowest_price < entry_px:
                profit_giveback_pct = ((lowest_price - exit_px) / lowest_price) * 100.0
            else:
                profit_giveback_pct = 0.0
                
        # Rekonsiliasi variabel internal pelacak
        if i < len(records):
            r = records[i]
            sl_final = r['Trailing_SL_Final'] / fractional_unit if fractional_unit > 0 else r['Trailing_SL_Final']
            atr_entry = r['ATR_At_Entry'] / fractional_unit if fractional_unit > 0 else r['ATR_At_Entry']
            exit_reason = r['Exit_Reason'] or ('End of Data' if exit_bar >= len(ohlc_df) - 1 else 'Trailing Stop Hit')
        else:
            sl_final = entry_px
            atr_entry = float(ohlc_df['ATR'].iloc[entry_bar])
            exit_reason = 'End of Data' if exit_bar >= len(ohlc_df) - 1 else 'Trailing Stop Hit'
            
        trade_rows.append({
            'Trade_ID': i + 1,
            'Mode': mode_label,
            'Position_Type': pos_type,
            'Size': round(pos_size, 4),
            'EntryTime': pd.to_datetime(tr['EntryTime']).strftime('%Y-%m-%d %H:%M'),
            'ExitTime': pd.to_datetime(tr['ExitTime']).strftime('%Y-%m-%d %H:%M'),
            'Duration': str(tr['Duration']),
            'Bars_Held': bars_held,
            'EntryPrice': round(entry_px, 2),
            'ExitPrice': round(exit_px, 2),
            'Trailing_SL_Final': round(sl_final, 2),
            'ATR_At_Entry': round(atr_entry, 4),
            'PnL_USD': round(pnl, 2),
            'Return_Pct': round(ret_pct, 2),
            'MAE_Pct': round(mae_pct, 2),
            'MFE_Pct': round(mfe_pct, 2),
            'Profit_Giveback_Pct': round(profit_giveback_pct, 2),
            'Exit_Reason': exit_reason
        })
        
    return pd.DataFrame(trade_rows)

print("Fungsi Rekonsiliasi Transaksi (reconcile_trades) berhasil didefinisikan.")

Fungsi Rekonsiliasi Transaksi (reconcile_trades) berhasil didefinisikan.


In [107]:
# =============================================================================
# CELL 6: METRIC EXTRACTION & BUY & HOLD BENCHMARK
# =============================================================================
def extract_metrics(stats, df, is_bnh=False):
    """
    Ekstraksi metrik kuantitatif terstandarisasi dan apple-to-apple:
      - Total Return (%)
      - CAGR (%)
      - Max Drawdown (%)
      - Calmar Ratio
      - Sharpe Ratio
      - Sortino Ratio
      - Win Rate (%)
      - Profit Factor
      - Total Trades
      - Exposure Time (%)
    """
    dur_years = (df.index[-1] - df.index[0]).total_seconds() / (365.25 * 86400)

    if is_bnh:
        tot_ret = float(stats['Buy & Hold Return [%]'])
        cagr = ((1.0 + tot_ret / 100.0) ** (1.0 / dur_years) - 1.0) * 100.0 if tot_ret > -100.0 else -100.0

        equity_series = df['Close'] / df['Close'].iloc[0]
        mdd = float(((equity_series - equity_series.cummax()) / equity_series.cummax()).min() * 100.0)

        daily_ret = df['Close'].resample('D').last().dropna().pct_change().dropna()
        wr = np.nan
        pf = np.nan
        n_trades = 1
        exposure = 100.0
    else:
        tot_ret = float(stats['Return [%]'])
        cagr = float(stats['Return (Ann.) [%]'])
        mdd = float(stats['Max. Drawdown [%]'])

        # Gunakan kurva ekuitas riil strategi untuk resampling harian yang identik dengan BnH
        eq_curve = stats['_equity_curve']['Equity']
        daily_ret = eq_curve.resample('D').last().dropna().pct_change().dropna()

        wr = float(stats['Win Rate [%]']) if not pd.isna(stats['Win Rate [%]']) else 0.0
        pf = float(stats['Profit Factor']) if not pd.isna(stats['Profit Factor']) else 0.0
        n_trades = int(stats['# Trades'])
        exposure = float(stats['Exposure Time [%]']) if not pd.isna(stats['Exposure Time [%]']) else 0.0

    # Perhitungan Sharpe, Sortino (Target Downside Deviation = 0), dan Calmar yang seragam
    if len(daily_ret) > 1 and daily_ret.std() > 0:
        sharpe = float((daily_ret.mean() / daily_ret.std()) * np.sqrt(252))
    else:
        sharpe = 0.0

    downside_diff = np.minimum(0.0, daily_ret)
    downside_std = np.sqrt(np.mean(downside_diff ** 2))
    if len(daily_ret) > 1 and downside_std > 0:
        sortino = float((daily_ret.mean() / downside_std) * np.sqrt(252))
    else:
        sortino = 0.0

    calmar = float(cagr / abs(mdd)) if abs(mdd) > 0 else 0.0

    return pd.Series({
        'Total Return (%)': tot_ret,
        'CAGR (%)': cagr,
        'Max Drawdown (%)': mdd,
        'Calmar Ratio': calmar,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Win Rate (%)': wr,
        'Profit Factor': pf,
        'Total Trades': n_trades,
        'Exposure Time (%)': exposure
    })

print("Fungsi Ekstraksi Metrik Kuantitatif (extract_metrics) berhasil didefinisikan.")

Fungsi Ekstraksi Metrik Kuantitatif (extract_metrics) berhasil didefinisikan.


In [108]:
# =============================================================================
# CELL 7: BATCH EXECUTION LOOP
# =============================================================================
ALL_PERFORMANCE_TABLES = {}
ALL_TRADES_TABLES = {}

print("=" * 80)
print(f"MEMULAI SIMULASI BACKTEST BATCH PADA INSTRUMEN: {ASSETS}")
print("=" * 80)

for asset in ASSETS:
    print(f"\n>>> Memproses Instrumen: [{asset}]")
    df = preprocess_asset_data(asset)
    print(f"    Data siap diuji: {len(df):,} bar 1H ({df.index[0]} s.d. {df.index[-1]})")
    
    # 1. Eksekusi Backtest Mode Long-Only
    bt_long = FractionalBacktest(
        df,
        SymmetricMTFStrategy,
        cash=GLOBAL_PARAMS['initial_capital'],
        commission=GLOBAL_PARAMS['fee'],
        trade_on_close=True,
        finalize_trades=True,
        fractional_unit=1e-4
    )
    stats_long = bt_long.run(mode='long_only')
    trades_long = reconcile_trades(stats_long, df, 'Long-Only')
    s_long = extract_metrics(stats_long, df, is_bnh=False)
    
    # 2. Eksekusi Evaluasi Buy & Hold Benchmark
    s_bnh = extract_metrics(stats_long, df, is_bnh=True)
    
    # 3. Eksekusi Backtest Mode Short-Only
    bt_short = FractionalBacktest(
        df,
        SymmetricMTFStrategy,
        cash=GLOBAL_PARAMS['initial_capital'],
        commission=GLOBAL_PARAMS['fee'],
        trade_on_close=True,
        finalize_trades=True,
        fractional_unit=1e-4
    )
    stats_short = bt_short.run(mode='short_only')
    trades_short = reconcile_trades(stats_short, df, 'Short-Only')
    s_short = extract_metrics(stats_short, df, is_bnh=False)
    
    # 4. Eksekusi Backtest Mode Combined (Dua Arah Riil)
    bt_comb = FractionalBacktest(
        df,
        SymmetricMTFStrategy,
        cash=GLOBAL_PARAMS['initial_capital'],
        commission=GLOBAL_PARAMS['fee'],
        trade_on_close=True,
        finalize_trades=True,
        fractional_unit=1e-4
    )
    stats_comb = bt_comb.run(mode='combined')
    trades_comb = reconcile_trades(stats_comb, df, 'Combined')
    s_comb = extract_metrics(stats_comb, df, is_bnh=False)
    
    # 5. Penyusunan Tabel Ringkasan Kinerja (Baris = Metrik, Kolom = Mode Pengujian)
    df_perf = pd.DataFrame({
        'Buy & Hold': s_bnh,
        'Long-Only': s_long,
        'Short-Only': s_short,
        'Combined': s_comb
    })
    ALL_PERFORMANCE_TABLES[asset] = df_perf
    
    # 6. Penggabungan Seluruh Log Transaksi Bersih untuk Instrumen Ini
    df_trades_all = pd.concat([trades_long, trades_short, trades_comb], ignore_index=True)
    df_trades_all['Trade_ID'] = range(1, len(df_trades_all) + 1)
    
    # Validasi Integritas Data (Zero NaN Guarantee)
    assert df_trades_all.isna().sum().sum() == 0, f"Ditemukan nilai NaN pada trades {asset}"
    assert len(df_trades_all.columns) == 18, f"Jumlah kolom tidak sesuai standar 18 kolom pada {asset}"
    
    ALL_TRADES_TABLES[asset] = df_trades_all
    
    print(f"    Simulasi [{asset}] selesai | Total Trades: Long={len(trades_long)}, Short={len(trades_short)}, Combined={len(trades_comb)} (Total: {len(df_trades_all)}, NaNs: 0)")

print("\n" + "=" * 80)
print("SELURUH SIMULASI BATCH INSTRUMEN BERHASIL DISELESAIKAN!")
print("=" * 80)

MEMULAI SIMULASI BACKTEST BATCH PADA INSTRUMEN: ['SPY', 'QQQ', 'DJI', 'AAPL']

>>> Memproses Instrumen: [SPY]
    Data siap diuji: 12,829 bar 1H (2018-02-19 16:00:00 s.d. 2024-05-10 16:00:00)


    Simulasi [SPY] selesai | Total Trades: Long=28, Short=17, Combined=45 (Total: 90, NaNs: 0)

>>> Memproses Instrumen: [QQQ]
    Data siap diuji: 12,769 bar 1H (2018-02-19 09:00:00 s.d. 2024-05-10 16:00:00)


    Simulasi [QQQ] selesai | Total Trades: Long=42, Short=27, Combined=69 (Total: 138, NaNs: 0)

>>> Memproses Instrumen: [DJI]
    Data siap diuji: 12,850 bar 1H (2018-02-19 09:00:00 s.d. 2024-05-10 16:00:00)


    Simulasi [DJI] selesai | Total Trades: Long=20, Short=13, Combined=33 (Total: 66, NaNs: 0)

>>> Memproses Instrumen: [AAPL]
    Data siap diuji: 11,880 bar 1H (2018-02-19 09:00:00 s.d. 2024-05-10 16:00:00)


    Simulasi [AAPL] selesai | Total Trades: Long=15, Short=8, Combined=23 (Total: 46, NaNs: 0)

SELURUH SIMULASI BATCH INSTRUMEN BERHASIL DISELESAIKAN!


In [109]:
# =============================================================================
# CELL 8: NOTEBOOK DISPLAY
# =============================================================================
from IPython.display import display

for asset in ASSETS:
    print("\n" + "=" * 85)
    print(f"TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - {asset}")
    print("=" * 85)
    
    df_display = ALL_PERFORMANCE_TABLES[asset].copy()
    
    # Format representasi angka rapi
    formatted_df = df_display.copy()
    for col in formatted_df.columns:
        formatted_df[col] = formatted_df[col].apply(
            lambda v: "-" if pd.isna(v) else (f"{int(v)}" if abs(v - round(v)) < 1e-6 and v >= 1 else f"{v:.2f}")
        )
        
    display(formatted_df)



TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - SPY


,Buy & Hold,Long-Only,Short-Only,Combined
Total Return (%),91.68,12.68,-21.95,-12.05
CAGR (%),11.03,1.89,-3.81,-1.99
Max Drawdown (%),-34.67,-21.24,-23.53,-28.92
Calmar Ratio,0.32,0.09,-0.16,-0.07
Sharpe Ratio,0.66,0.24,-0.51,-0.11
Sortino Ratio,0.91,0.33,-0.67,-0.15
Win Rate (%),-,75,17.65,53.33
Profit Factor,-,1.40,0.33,0.85
Total Trades,1,28,17,45
Exposure Time (%),100,39.78,5.90,45.68



TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - QQQ


,Buy & Hold,Long-Only,Short-Only,Combined
Total Return (%),167.51,80.93,-31.43,24.06
CAGR (%),17.14,9.80,-5.78,3.46
Max Drawdown (%),-37.25,-27.62,-38.48,-36.32
Calmar Ratio,0.46,0.35,-0.15,0.10
Sharpe Ratio,0.80,0.78,-0.44,0.28
Sortino Ratio,1.14,1.09,-0.55,0.37
Win Rate (%),-,83.33,25.93,60.87
Profit Factor,-,2.42,0.43,1.27
Total Trades,1,42,27,69
Exposure Time (%),100,53.68,8.90,62.59



TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - DJI


,Buy & Hold,Long-Only,Short-Only,Combined
Total Return (%),56.42,19.12,-21.08,-6.00
CAGR (%),7.46,2.78,-3.64,-0.96
Max Drawdown (%),-37.63,-11.19,-25.98,-24.60
Calmar Ratio,0.20,0.25,-0.14,-0.04
Sharpe Ratio,0.48,0.39,-0.45,-0.03
Sortino Ratio,0.67,0.56,-0.61,-0.05
Win Rate (%),-,85,7.69,54.55
Profit Factor,-,2.25,0.22,0.89
Total Trades,1,20,13,33
Exposure Time (%),100,31.50,5.18,36.68



TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - AAPL


,Buy & Hold,Long-Only,Short-Only,Combined
Total Return (%),327.63,105.17,-39.86,23.39
CAGR (%),26.31,12.98,-8.27,3.63
Max Drawdown (%),-37.42,-31.43,-51.95,-44.57
Calmar Ratio,0.70,0.41,-0.16,0.08
Sharpe Ratio,1.15,0.82,-0.65,0.28
Sortino Ratio,1.67,1.23,-0.89,0.40
Win Rate (%),-,53.33,25,43.48
Profit Factor,-,3.00,0.27,1.45
Total Trades,1,15,8,23
Exposure Time (%),100,56.26,17.28,73.54


In [110]:
# =============================================================================
# CELL 9: DUAL EXCEL EXPORTER (openpyxl)
# =============================================================================
# Inisialisasi Workbook Excel Terpisah
wb_ringkasan = openpyxl.Workbook()
wb_ringkasan.remove(wb_ringkasan.active) # Hapus sheet default

wb_trades = openpyxl.Workbook()
wb_trades.remove(wb_trades.active) # Hapus sheet default

# Gaya Visual Terstandarisasi (Institutional Clean Theme)
header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
cell_font = Font(name="Calibri", size=10)
bold_font = Font(name="Calibri", size=10, bold=True)
thin_border = Border(
    left=Side(style='thin', color='D9D9D9'),
    right=Side(style='thin', color='D9D9D9'),
    top=Side(style='thin', color='D9D9D9'),
    bottom=Side(style='thin', color='D9D9D9')
)

# -----------------------------------------------------------------------------
# 1. Ekspor Berkas Ringkasan Kinerja (Ringkasan_Kinerja_Semua_Instrumen.xlsx)
# -----------------------------------------------------------------------------
for asset in ASSETS:
    ws = wb_ringkasan.create_sheet(title=asset)
    df_perf = ALL_PERFORMANCE_TABLES[asset]
    
    # Baris Judul Kolom
    headers = ['Metrik'] + df_perf.columns.tolist()
    ws.append(headers)
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border
        
    # Baris Data Metrik
    for row_idx, (metric_name, row_vals) in enumerate(df_perf.iterrows(), start=2):
        row_data = [metric_name]
        for val in row_vals:
            if pd.isna(val):
                row_data.append("-")
            elif metric_name == 'Total Trades':
                row_data.append(int(val))
            else:
                row_data.append(round(float(val), 2))
        ws.append(row_data)
        
        # Penataan Gaya Sel Data
        metric_cell = ws.cell(row=row_idx, column=1)
        metric_cell.font = bold_font
        metric_cell.border = thin_border
        
        for c_idx in range(2, len(row_data) + 1):
            c = ws.cell(row=row_idx, column=c_idx)
            c.font = cell_font
            c.alignment = Alignment(horizontal="right", vertical="center")
            c.border = thin_border
            if isinstance(c.value, (int, float)):
                c.number_format = '#,##0.00' if isinstance(c.value, float) else '#,##0'
                
    # Otomatisasi Penyesuaian Lebar Kolom
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 4, 15)

wb_ringkasan.save(FILE_RINGKASAN)
print(f"Laporan 1 Berhasil Disimpan: {FILE_RINGKASAN}")

# -----------------------------------------------------------------------------
# 2. Ekspor Berkas Detail Transaksi (Detail_Trades_Semua_Instrumen.xlsx)
# -----------------------------------------------------------------------------
for asset in ASSETS:
    ws = wb_trades.create_sheet(title=f"Trades_{asset}")
    df_tr = ALL_TRADES_TABLES[asset]
    
    # Validasi Integritas Data Sebelum Ekspor
    assert df_tr.isna().sum().sum() == 0, f"Ditemukan nilai NaN pada trades {asset} sebelum ekspor"
    
    headers = df_tr.columns.tolist()
    ws.append(headers)
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border
        
    for row_idx, (_, tr_row) in enumerate(df_tr.iterrows(), start=2):
        ws.append(tr_row.tolist())
        for c_idx in range(1, len(headers) + 1):
            c = ws.cell(row=row_idx, column=c_idx)
            c.font = cell_font
            c.border = thin_border
            col_name = headers[c_idx - 1]
            
            if col_name in ['EntryPrice', 'ExitPrice', 'Trailing_SL_Final', 'PnL_USD', 'Return_Pct', 'MAE_Pct', 'MFE_Pct', 'Profit_Giveback_Pct']:
                c.alignment = Alignment(horizontal="right", vertical="center")
                if isinstance(c.value, (int, float)):
                    c.number_format = '#,##0.00'
            elif col_name in ['Size', 'ATR_At_Entry']:
                c.alignment = Alignment(horizontal="right", vertical="center")
                if isinstance(c.value, (int, float)):
                    c.number_format = '#,##0.0000'
            elif col_name in ['Trade_ID', 'Bars_Held']:
                c.alignment = Alignment(horizontal="center", vertical="center")
                if isinstance(c.value, (int, float)):
                    c.number_format = '#,##0'
            else:
                c.alignment = Alignment(horizontal="center", vertical="center")

    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 14)

wb_trades.save(FILE_TRADES)
print(f"Laporan 2 Berhasil Disimpan: {FILE_TRADES}")
print("\nSeluruh proses ekspor Excel selesai.")

Laporan 1 Berhasil Disimpan: d:\Kuliah\TA\ta\Ringkasan_Kinerja_Semua_Instrumen.xlsx
Laporan 2 Berhasil Disimpan: d:\Kuliah\TA\ta\Detail_Trades_Semua_Instrumen.xlsx

Seluruh proses ekspor Excel selesai.
